# Visualise distributions of target and diagnosis

In [ ]:
# Imports
import h5py
import numpy as np
import matplotlib.pyplot as plt
from skrub import TableReport
import pandas as pd


In [ ]:
# Import dataset and remove images. Keep only tabular data

input_file = r'path\to\your\dataset.h5'  # Change this to the path of the dataset to explore

rows = []

with h5py.File(input_file, "r") as f:
    for fold_name in f.keys():   # fold_0 ... fold_5
        fold = f[fold_name]

        targets = fold["target"][:]
        diagnoses = fold["diagnosis"][:]
        patient_idxs = fold["patient_idx"][:]

        for t, d, p in zip(targets, diagnoses, patient_idxs):
            rows.append({
                "fold": fold_name,
                "patient_idx": p,
                "diagnosis": d,
                "target": t
            })

df = pd.DataFrame(rows)


In [ ]:
TableReport(df)

In [ ]:
# Show how many unique patients have which diagnosis (Degree of Dysplasia)
patient_diagnosis = df.groupby("patient_idx")["diagnosis"].first()
patient_diagnosis.value_counts()


In [ ]:
# show total number of unique patients
total_diagnosed_patients = patient_diagnosis.value_counts().sum()
print(f"Total unique patients: {total_diagnosed_patients}")

# show total number of samples in the dataset
total_samples = len(df)
print(f"Total samples: {total_samples}")

In [ ]:
# show how many unique patients have which target (Dysplasia(1) or not(0))
patient_target = df.groupby("patient_idx")["target"].first()
patient_target.value_counts()

In [ ]:
# Make tabel displaying distribution of target per fold
Dist = df.groupby('fold')[['target']].value_counts()
Dist_table = Dist.reset_index(name='count')
Dist_table

In [ ]:
# Make a bar chart for each fold showing the distribution of target values
import matplotlib.colors as mcolors

counts = (
    df.groupby(['fold', 'target'])
      .size()
      .unstack(fill_value=0)
      .reindex(columns=[0, 1], fill_value=0)
      .sort_index()
)

x = np.arange(len(counts.index))
width = 0.4

plt.figure(figsize=(10, 6))
plt.bar(x - width/2, counts[0], width=width, label='0 = Ingen AD', color='seagreen')
plt.bar(x + width/2, counts[1], width=width, label='1 = AD', color='darkmagenta')

plt.xticks(x, counts.index)
plt.xlabel('Fold #')
plt.ylabel('Antall prøver')
plt.title('Fordeling av AD (target) per fold')
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
# Make tabel displaying distribution of diagnosis per fold
Dist = df.groupby('fold')[['diagnosis']].value_counts()
Dist_table = Dist.reset_index(name='count')
Dist_table

In [ ]:
# Make correlation matrix
correlation_matrix = df.corr(numeric_only=True)
plt.matshow(correlation_matrix, cmap='viridis')
plt.colorbar()
plt.xticks(range(len(correlation_matrix.columns)), correlation_matrix.columns, rotation=90)
plt.yticks(range(len(correlation_matrix.columns)), correlation_matrix.columns)
plt.title('Correlation Matrix')
plt.show()